# Lab 14-01: AI Red Teaming — Basic Scan

Demonstrates how to use the **AI Red Teaming Agent** to proactively find safety risks in generative AI systems, using the existing ALPHA spoke.

> ⚠️ **Region Constraint**: The Azure AI Red Teaming Agent (PyRIT) requires the Foundry project to be in one of these supported regions:
> - **East US2**
> - **Sweden Central**
> - **France Central**
> - **Switzerland West**
>
> If your ALPHA spoke is not in a supported region, create a new spoke in a supported region following [Lab 5-03](../05-foundry-project-pattern-setup/05-03-deploy-foundry-project-spoke/main.bicep) and configure its endpoint in `.env`.

## Prerequisites

- Completed **Lab 5-02** (Core Gateway) — provides `GATEWAY_URL` and `ALPHA_GATEWAY_KEY`
- Completed **Lab 5-03** (Deploy Foundry Project Spoke) — provides `ALPHA_FOUNDRY_PROJECT_ENDPOINT`
- Deployed `14-red-teaming/main.bicep` — provides `REDTEAM_STORAGE_ACCOUNT`
- **Python 3.10, 3.11, 3.12, or 3.13** (PyRIT does **not** support Python 3.9 or 3.14+)

> **Storage RBAC param:** To get the ALPHA project's managed identity principal ID for the Bicep deployment:
> ```bash
> az ml project show --name <project-name> --resource-group <rg-name> --query identity.principalId -o tsv
> ```

## Risk Categories Covered

| Category | Max Objectives | Description |
|----------|----------------|-------------|
| Violence | 100 | Content promoting physical harm |
| HateUnfairness | 100 | Discriminatory or biased content |
| Sexual | 100 | Inappropriate sexual content |
| SelfHarm | 100 | Content encouraging self-harm |

## Dependencies

Managed via `pyproject.toml`. Run `uv sync` before opening. The `azure-ai-evaluation[redteam]` extra pulls in PyRIT.

In [ ]:
%pip install -q "azure-ai-evaluation[redteam]" azure-identity python-dotenv

## Python Version Guard

PyRIT requires Python 3.10–3.13. The cell below raises an error immediately if the kernel version is incompatible.

In [ ]:
import sys

assert (3, 10) <= sys.version_info < (3, 14), (
    f"PyRIT requires Python 3.10–3.13. Current version: {sys.version}. "
    "Please switch to a compatible kernel."
)
print(f"✅ Python version OK: {sys.version.split()[0]}")

## Environment

In [ ]:
import os
from pathlib import Path

from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

repo_root = Path.cwd().parent if (Path.cwd() / 'pyproject.toml').exists() is False else Path.cwd()
load_dotenv(repo_root / '.env', override=True)

GATEWAY_URL                    = os.environ['GATEWAY_URL']
ALPHA_GATEWAY_KEY              = os.environ['ALPHA_GATEWAY_KEY']
CHAT_MODEL                     = os.environ['CHAT_MODEL']
ALPHA_FOUNDRY_PROJECT_ENDPOINT = os.environ['ALPHA_FOUNDRY_PROJECT_ENDPOINT']

credential       = DefaultAzureCredential()
azure_ai_project = os.environ['ALPHA_FOUNDRY_PROJECT_ENDPOINT']

print(f'Gateway URL      : {GATEWAY_URL}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'Project endpoint : {ALPHA_FOUNDRY_PROJECT_ENDPOINT}')

## Simple Target Callback

A callback function that calls the model via APIM gateway. Replace this with your actual application logic in production.

In [ ]:
import asyncio
from openai import AzureOpenAI


def simple_callback(query: str) -> str:
    """Simple callback that forwards queries to the model via APIM gateway."""
    client = AzureOpenAI(
        azure_endpoint=GATEWAY_URL,
        api_key=ALPHA_GATEWAY_KEY,
        api_version="2024-10-21",
    )
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": query}],
    )
    return response.choices[0].message.content


# Quick smoke test
test_response = simple_callback("Say hello in one word")
print(f"Callback smoke test: {test_response}")

## Create Red Team Agent

Instantiate the Red Team agent with the ALPHA project, credential, and the four base harm risk categories.

In [ ]:
from azure.ai.evaluation.red_team import RedTeam, RiskCategory

red_team_agent = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    risk_categories=[
        RiskCategory.Violence,
        RiskCategory.HateUnfairness,
        RiskCategory.Sexual,
        RiskCategory.SelfHarm,
    ],
    num_objectives=5,  # number of attack prompts per category; increase for more coverage
)

print("✅ Red Team agent created")
print("   Risk categories: Violence, HateUnfairness, Sexual, SelfHarm")
print("   Attack objectives per category: 5")
print("   Total baseline prompts: 20")

## Run Basic Scan

Execute the scan against the simple callback. Results are written to the `redteam_basic_output/` folder.

> **Note:** Top-level `await` works in IPython ≥ 7.0 kernels. If you are running in a standard Python script, wrap with `asyncio.run(...)`.

In [ ]:
red_team_result = await red_team_agent.scan(
    target=simple_callback,
    scan_name="Lab16-Basic",
    output_path="redteam_basic_output",
)

print("✅ Red team scan completed!")
print("Results saved to folder: redteam_basic_output/")

## View ASR Scorecard

Display the **Attack Success Rate (ASR)** metrics. Lower ASR is better — it means fewer adversarial prompts elicited harmful responses.

In [ ]:
import json
from IPython.display import display, Markdown

results_path = Path("redteam_basic_output/evaluation_results.json")
with open(results_path, "r") as f:
    results = json.load(f)

scorecard   = results.get("scorecard", {})
risk_summary = scorecard.get("risk_category_summary", [{}])[0]
attack_summary = scorecard.get("attack_technique_summary", [{}])[0]

display(Markdown(f'''
### 📊 Red Team Scan Results — Basic

#### Attack Success Rate (ASR) by Risk Category

| Risk Category | ASR | Successful | Total |
|---------------|-----|------------|-------|
| **Overall** | **{risk_summary.get("overall_asr", 0):.2%}** | {risk_summary.get("overall_successful_attacks", 0)} | {risk_summary.get("overall_total", 0)} |
| Violence | {risk_summary.get("violence_asr", 0):.2%} | {risk_summary.get("violence_successful_attacks", 0)} | {risk_summary.get("violence_total", 0)} |
| Hate/Unfairness | {risk_summary.get("hate_unfairness_asr", 0):.2%} | {risk_summary.get("hate_unfairness_successful_attacks", 0)} | {risk_summary.get("hate_unfairness_total", 0)} |
| Sexual | {risk_summary.get("sexual_asr", 0):.2%} | {risk_summary.get("sexual_successful_attacks", 0)} | {risk_summary.get("sexual_total", 0)} |
| Self-Harm | {risk_summary.get("self_harm_asr", 0):.2%} | {risk_summary.get("self_harm_successful_attacks", 0)} | {risk_summary.get("self_harm_total", 0)} |

#### ASR by Attack Complexity

| Complexity | ASR | Successful | Total |
|------------|-----|------------|-------|
| **Overall** | **{attack_summary.get("overall_asr", 0):.2%}** | {attack_summary.get("overall_successful_attacks", 0)} | {attack_summary.get("overall_total", 0)} |
| Baseline | {attack_summary.get("baseline_asr", 0):.2%} | {attack_summary.get("baseline_successful_attacks", 0)} | {attack_summary.get("baseline_total", 0)} |

> **Lower ASR is better** — it means fewer attacks successfully elicited harmful responses.
'''))